# S2 · Aço e minério no comércio exterior do Brasil

**Pergunta:** que produtos de aço, minério e alumínio o Brasil está vendendo e comprando lá fora, de onde, para onde e a que preço? E quanto o aço importado está ocupando do mercado?

Fonte: Comex Stat (MDIC), API oficial, exportação e importação mensal de janeiro de 2015 a outubro de 2026, por posição do Sistema Harmonizado (SH4), país e estado. Valores em US$ FOB e quilos líquidos.

## 1. Carregar
`baixar_dados.py` consulta a API e grava dois arquivos: produto × país × mês e produto × estado × mês. 2026 vai até outubro, então toda comparação com 2026 usa **janeiro a outubro** dos dois anos.

In [1]:
import pandas as pd, numpy as np, json
pais = pd.read_csv('dados/pais.csv', dtype={'sh4': str})
uf = pd.read_csv('dados/uf.csv', dtype={'sh4': str})
for n, d in [('país', pais), ('estado', uf)]:
    print(f"{n}: {len(d):,} linhas · {d.ano.min()}-01 a {d.ano.max()}-{d[d.ano == d.ano.max()].mes.max():02d} · {d.sh4.nunique()} posições SH4")

país: 148,363 linhas · 2015-01 a 2026-10 · 30 posições SH4
estado: 65,087 linhas · 2015-01 a 2026-10 · 30 posições SH4


## 2. Agrupar as posições em produtos
O SH4 é detalhado demais para uma leitura de mercado. Agrupo em 10 produtos, pela forma do aço (plano, longo, tubo) e pela etapa (minério, gusa, ferroliga, semiacabado).

In [2]:
GRUPO = {'2601': 'Minério de ferro', '7201': 'Ferro-gusa', '7202': 'Ferroligas', '7207': 'Semiacabados'}
GRUPO.update({s: 'Aço plano' for s in ['7208', '7209', '7210', '7211', '7212', '7219', '7220', '7225', '7226']})
GRUPO.update({s: 'Aço longo' for s in ['7213', '7214', '7215', '7216', '7217', '7221', '7222', '7227', '7228']})
GRUPO.update({s: 'Tubos de aço' for s in ['7304', '7305', '7306']})
GRUPO.update({'7601': 'Alumínio bruto'}); GRUPO.update({s: 'Alumínio laminado e extrudado' for s in ['7604', '7605', '7606', '7607']})
for d in (pais, uf): d['produto'] = d.sh4.map(GRUPO)
print(pais.produto.isna().sum(), 'posições sem grupo')
ano = pais.groupby(['produto', 'fluxo', 'ano'])[['kg', 'fob_usd']].sum()
ano['mil_t'] = ano.kg / 1e6; ano['usd_t'] = ano.fob_usd / ano.kg * 1000
print(ano.mil_t.unstack(['fluxo']).unstack('ano').round(0)[[('export', 2019), ('export', 2025), ('import', 2019), ('import', 2025)]].to_string())

0 posições sem grupo
fluxo                            export            import        
ano                                2019      2025    2019    2025
produto                                                          
Alumínio bruto                    134.0     323.0   376.0   219.0
Alumínio laminado e extrudado     112.0     156.0   207.0   230.0
Aço longo                        1347.0     862.0   443.0  1010.0
Aço plano                        2201.0     987.0  1358.0  4092.0
Ferro-gusa                       2905.0    4056.0     0.0     0.0
Ferroligas                        539.0     660.0    87.0    94.0
Minério de ferro               350960.0  416250.0   298.0     0.0
Semiacabados                     6777.0    7399.0   199.0   642.0
Tubos de aço                      558.0     414.0   118.0   221.0


## 3. O aço plano virou importação líquida
Aço plano (bobina a quente e a frio, chapa, galvanizado, inox plano) é o produto que abastece a indústria de transformação. Saldo = exportado − importado, em toneladas.

In [3]:
pl = ano.loc['Aço plano'].mil_t.unstack('fluxo')
pl['saldo'] = pl.export - pl['import']
print(pl.round(0).to_string())
i19, i25, e19, e25 = pl.loc[2019, 'import'], pl.loc[2025, 'import'], pl.loc[2019, 'export'], pl.loc[2025, 'export']
print(f"\n2019: exportou {e19/1e3:.1f} Mt, importou {i19/1e3:.1f} Mt · saldo {pl.loc[2019,'saldo']/1e3:+.1f} Mt")
print(f"2025: exportou {e25/1e3:.1f} Mt, importou {i25/1e3:.1f} Mt · saldo {pl.loc[2025,'saldo']/1e3:+.1f} Mt")
print(f"importação 2025 ÷ 2019 = {i25/i19:.1f} vezes")

fluxo  export  import   saldo
ano                          
2015   3476.0  1839.0  1636.0
2016   3111.0   904.0  2207.0
2017   3430.0  1532.0  1899.0
2018   2514.0  1505.0  1010.0
2019   2201.0  1358.0   843.0
2020   1422.0  1258.0   164.0
2021   1628.0  2404.0  -776.0
2022   2231.0  2011.0   220.0
2023   1226.0  2836.0 -1611.0
2024    890.0  3140.0 -2250.0
2025    987.0  4092.0 -3104.0
2026    822.0  2909.0 -2087.0

2019: exportou 2.2 Mt, importou 1.4 Mt · saldo +0.8 Mt
2025: exportou 1.0 Mt, importou 4.1 Mt · saldo -3.1 Mt
importação 2025 ÷ 2019 = 3.0 vezes


## 4. De onde vem o aço plano importado
Participação de cada origem no volume, janeiro a outubro de cada ano (para comparar com 2026), e o preço médio em US$ por tonelada.

In [4]:
ip = pais[(pais.produto == 'Aço plano') & (pais.fluxo == 'import') & (pais.mes <= 10)]
tab = {}
for a in [2019, 2023, 2024, 2025, 2026]:
    q = ip[ip.ano == a].groupby('pais').agg(kg=('kg', 'sum'), fob=('fob_usd', 'sum'))
    tab[a] = q
    q = q.assign(part=q.kg / q.kg.sum(), usd_t=q.fob / q.kg * 1000).sort_values('kg', ascending=False)
    print(a, f"total {q.kg.sum()/1e9:.2f} Mt (jan–out)"); print(q.head(5)[['part', 'usd_t']].round(3).to_string(), '\n')
t25, t26 = tab[2025].kg.sum(), tab[2026].kg.sum()
ch25, ch26 = tab[2025].kg['China'] / t25, tab[2026].kg['China'] / t26
vn25, vn26 = tab[2025].kg.get('Vietnã', 0) / t25, tab[2026].kg.get('Vietnã', 0) / t26
print(f"jan–out 2026 × 2025: importação {t26/t25-1:+.0%} · China {ch25:.0%} → {ch26:.0%} · Vietnã {vn25:.0%} → {vn26:.0%}")

2019 total 1.21 Mt (jan–out)
                part    usd_t
pais                         
China          0.531  735.664
Coreia do Sul  0.121  986.434
Rússia         0.056  572.640
Ucrânia        0.033  520.035
Áustria        0.032  819.342 



2023 total 2.29 Mt (jan–out)
                part     usd_t
pais                          
China          0.712   893.986
Coreia do Sul  0.108   875.248
Áustria        0.044   988.923
Alemanha       0.018  1866.768
Suécia         0.015  2311.444 

2024 total 2.71 Mt (jan–out)
                part     usd_t
pais                          
China          0.811   734.900
Coreia do Sul  0.055  1181.650
Japão          0.023  1196.606
Alemanha       0.019  1548.384
Suécia         0.012  2024.976 

2025 total 3.50 Mt (jan–out)
                part     usd_t
pais                          
China          0.692   694.291
Coreia do Sul  0.161   709.581
Egito          0.031   574.922
Alemanha       0.014  1720.148
Japão          0.013  1600.360 



2026 total 2.91 Mt (jan–out)
                part    usd_t
pais                         
China          0.424  695.497
Coreia do Sul  0.175  710.338
Vietnã         0.162  735.577
Egito          0.042  585.965
Áustria        0.034  940.276 

jan–out 2026 × 2025: importação -17% · China 69% → 42% · Vietnã 1% → 16%


## 5. Preço: o importado chega mais barato que o exportado
Preço médio FOB por tonelada do aço plano. FOB do importado não inclui frete, seguro, imposto de importação nem internação, então o custo posto no Brasil é maior.

In [5]:
pr = ano.loc['Aço plano'].usd_t.unstack('fluxo')
ipa = pais[(pais.produto == 'Aço plano') & (pais.fluxo == 'import')]
c = ipa.assign(origem=np.where(ipa.pais == 'China', 'China', 'outras')).groupby(['ano', 'origem']).agg(kg=('kg', 'sum'), fob=('fob_usd', 'sum'))
pr = pr.join((c.fob / c.kg * 1000).unstack('origem').add_prefix('import_'))
print(pr.round(0).to_string())

      export  import  import_China  import_outras
ano                                              
2015   549.0   794.0         643.0          987.0
2016   513.0   780.0         549.0         1045.0
2017   643.0   822.0         670.0          959.0
2018   718.0   941.0         764.0         1124.0
2019   645.0   901.0         744.0         1077.0
2020   624.0   890.0         709.0         1097.0
2021   978.0   984.0         899.0         1162.0
2022  1070.0  1317.0        1183.0         1594.0
2023  1109.0   997.0         867.0         1342.0
2024   999.0   866.0         735.0         1405.0
2025   891.0   789.0         693.0          996.0
2026   900.0   800.0         695.0          877.0


## 6. O número

In [6]:
print(f"Em 2025, o Brasil importou {i25/1e3:.1f} milhões de toneladas de aço plano, {i25/i19:.1f} vezes o volume de 2019,")
print(f"e exportou {e25/1e3:.1f} milhão: o saldo foi de {pl.loc[2019,'saldo']/1e3:+.1f} Mt em 2019 para {pl.loc[2025,'saldo']/1e3:+.1f} Mt em 2025.")
print(f"Em 2026 (jan–out), a importação caiu {abs(t26/t25-1):.0%}, mas a origem mudou: China de {ch25:.0%} para {ch26:.0%}, Vietnã de {vn25:.0%} para {vn26:.0%}.")

Em 2025, o Brasil importou 4.1 milhões de toneladas de aço plano, 3.0 vezes o volume de 2019,
e exportou 1.0 milhão: o saldo foi de +0.8 Mt em 2019 para -3.1 Mt em 2025.
Em 2026 (jan–out), a importação caiu 17%, mas a origem mudou: China de 69% para 42%, Vietnã de 1% para 16%.


## 7. Exportar para o painel
O mesmo `exportar.py` que a atualização automática usa: gera `painel/dados.js` (produto × país × mês e produto × estado × mês, em toneladas e mil US$). O painel recalcula tudo no navegador.

In [7]:
from exportar import exportar
exportar()

dados.js gravado · até 10/2026 · 71469 linhas por país · 26056 por estado · 1916 KB


True

## 8. Limitações
- **O dado mostra o quê, não o porquê.** A queda da importação em 2026 e a troca de origem coincidem com medidas de defesa comercial e com mudanças de preço internacional, mas este dado sozinho não separa as causas.
- **FOB não é o preço posto na fábrica.** No importado faltam frete, seguro, imposto de importação, AFRMM e internação; no exportado, o preço é na saída do porto.
- **Toneladas de produtos diferentes não se somam.** O agrupamento junta bobina a quente, a frio, galvanizado e inox, com preços muito diferentes. A mistura muda o preço médio do grupo mesmo sem mudar o preço de nenhum item.
- **País de origem é o declarado na importação.** Aço processado num terceiro país (relaminado ou revestido) aparece com a origem desse país, o que importa para ler a troca da China pelo Vietnã.
- **Estado é a UF do produto** (onde foi produzido, na exportação, ou o domicílio do importador), não necessariamente onde o aço é consumido.
- **Comércio exterior não é consumo aparente.** Para saber a fatia do importado no mercado brasileiro falta a produção e as vendas internas (Aço Brasil), que não estão nesta base.
- **2026 vai até outubro.** Toda comparação com 2026 usa janeiro a outubro dos dois anos; os dados mais recentes podem ser revisados pelo MDIC.